# Mirror Test — Colab GPU driver

Runs the GPU phases of the pipeline (generation → paraphrase → judging → perplexity baseline).

**Before starting:** Runtime → Change runtime type → **T4 GPU**.

**How to use:** run the SETUP cells top to bottom once per session, then run whichever STAGE cell this session is for (one model per session is a good rhythm — see `docs/04_colab_kaggle_guide.md`). Every script resumes automatically: if Colab disconnects, reconnect, re-run SETUP, re-run the same stage cell.

**At the end of EVERY session:** run the last cell (commit + push results). Never leave results only in Colab.

In [ ]:
# SETUP 1/4 — confirm a GPU is attached (you should see a T4 table)
!nvidia-smi

In [ ]:
# SETUP 2/4 — get the repo. EDIT the URL to your GitHub username.
REPO_URL = "https://github.com/YOUR-USERNAME/mirror-test-llms.git"  # <-- EDIT
import os
if not os.path.exists("mirror-test-llms"):
    !git clone {REPO_URL}
%cd mirror-test-llms
!git pull

In [ ]:
# SETUP 3/4 — install (Colab already ships torch)
!pip -q install -U transformers accelerate bitsandbytes datasets sentence-transformers pyyaml
import torch; print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# SETUP 4/4 — Hugging Face login (needed for the gated Llama & Gemma foils).
# One-time prerequisite: accept the licenses on both model pages while
# logged in at huggingface.co (see docs/04_colab_kaggle_guide.md §2).
from huggingface_hub import login
login()  # paste a READ token when prompted

## STAGE cells — run the one this session is for
(Prompts must already be frozen and committed: `python src/00_build_prompts.py` — CPU, can be done on your laptop or in a cell here.)

In [ ]:
# STAGE A — GENERATION (~20 min for 0.5B ... ~2 h for 7B on a T4)
# One judge per session; judges need --placebo. Uncomment ONE line:
!python src/01_generate.py --models qwen2.5-0.5b-instruct --placebo
#!python src/01_generate.py --models qwen2.5-1.5b-instruct --placebo
#!python src/01_generate.py --models qwen2.5-3b-instruct   --placebo
#!python src/01_generate.py --models qwen2.5-7b-instruct   --placebo
#!python src/01_generate.py --models qwen2.5-14b-instruct  --placebo   # prefer Kaggle 2xT4
#!python src/01_generate.py --models llama-3.2-3b-instruct
#!python src/01_generate.py --models gemma-2-9b-it
#!python src/01_generate.py --models mistral-7b-instruct-v0.3

In [ ]:
# STAGE B — PAIR BUILDING (CPU-fast; run once after ALL generation is done)
!python src/02_build_pairs.py
# then the paraphrase materials (GPU, ~1 h):
!python src/02b_paraphrase.py
# if you dropped the 14B judge, use: --judge qwen2.5-7b-instruct
# (and set paraphrase.judge accordingly in configs/models.yaml)

In [ ]:
# STAGE C — PPP JUDGING (the mirror test). One judge per session; uncomment ONE:
!python src/03_judge_ppp.py --judge qwen2.5-0.5b-instruct --include-placebo
#!python src/03_judge_ppp.py --judge qwen2.5-1.5b-instruct --include-placebo
#!python src/03_judge_ppp.py --judge qwen2.5-3b-instruct   --include-placebo
#!python src/03_judge_ppp.py --judge qwen2.5-7b-instruct   --include-placebo
#!python src/03_judge_ppp.py --judge qwen2.5-14b-instruct  --include-placebo

# MAIN CELL extras (largest judge only): all 3 phrasings + paraphrase condition
#!python src/03_judge_ppp.py --judge qwen2.5-14b-instruct --foils llama-3.2-3b-instruct --phrasings 0 1 2 --include-paraphrase

# base-vs-instruct ablation (§9.5):
#!python src/03_judge_ppp.py --judge qwen2.5-7b-base
#!python src/03_judge_ppp.py --judge qwen2.5-14b-base

In [ ]:
# STAGE D — IPP judging (~10 min per judge; can batch several in one session)
for judge in ["qwen2.5-0.5b-instruct", "qwen2.5-1.5b-instruct",
              "qwen2.5-3b-instruct", "qwen2.5-7b-instruct",
              "qwen2.5-14b-instruct"]:
    !python src/04_judge_ipp.py --judge {judge}

In [ ]:
# STAGE E — perplexity baseline (GPU; one judge per run, resumable)
!python src/05_baselines.py perplexity --judge qwen2.5-0.5b-instruct
#!python src/05_baselines.py perplexity --judge qwen2.5-1.5b-instruct
#!python src/05_baselines.py perplexity --judge qwen2.5-3b-instruct
#!python src/05_baselines.py perplexity --judge qwen2.5-7b-instruct
#!python src/05_baselines.py perplexity --judge qwen2.5-14b-instruct --include-paraphrase
#!python src/05_baselines.py perplexity --judge qwen2.5-7b-base      # §9.5 proxy
#!python src/05_baselines.py perplexity --judge qwen2.5-14b-base

In [ ]:
# STAGE F — analysis (CPU; also fine on your laptop)
!pip -q install scikit-learn matplotlib
!python src/05_baselines.py stylometric
!python src/06_stats.py
from IPython.display import Image, display
display(Image("results/figures/fig1_scale_curve.png"))
display(Image("results/figures/fig2_paraphrase.png"))

In [ ]:
# ALWAYS LAST — checkpoint everything to GitHub before the session dies.
# For the password prompt use a GitHub Personal Access Token (repo scope).
!git config user.email "you@example.com" && git config user.name "Your Name"   # <-- EDIT once
!git add -A && git commit -m "colab session results" || echo 'nothing new'
!git push